In [2]:
import sys
from pathlib import Path

import torch
from torch.utils.data import DataLoader

# Add VisionInspect project root to Python path
PROJECT_ROOT = Path(
    r"X:\VScode\Artificial_intelligence_n_Machine_learning\VisionInspect"
)

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.data.dataset_loader import MVTecDataset
from src.models.autoencoder import Autoencoder
from src.detection.anomaly_score import reconstruction_loss


# -----------------------------
# Configuration
# -----------------------------

DATASET_ROOT = (
    PROJECT_ROOT
    / "dataset"
    / "mvtec_anomaly_detection"
)

CATEGORY = "bottle"
BATCH_SIZE = 16
EPOCHS = 5
LEARNING_RATE = 1e-3


# -----------------------------
# Device
# -----------------------------

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("Device:", device)
print("Category:", CATEGORY)


# -----------------------------
# Dataset
# -----------------------------

dataset = MVTecDataset(
    dataset_root=DATASET_ROOT,
    category=CATEGORY
)

train_loader = DataLoader(
    dataset,
    batch_size=BATCH_SIZE,
    shuffle=True
)

print("Training images:", len(dataset))
print("Batches per epoch:", len(train_loader))


# -----------------------------
# Model
# -----------------------------

model = Autoencoder().to(device)

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=LEARNING_RATE
)

print("Model ready.")

Device: cuda
Category: bottle
Training images: 209
Batches per epoch: 14
Model ready.


In [3]:
# -----------------------------
# Training loop
# -----------------------------

model.train()

for epoch in range(EPOCHS):
    epoch_loss = 0.0

    for images in train_loader:
        images = images.to(device)

        # Forward pass
        reconstructed = model(images)

        # Calculate reconstruction loss
        loss = reconstruction_loss(
            images,
            reconstructed
        )

        # Backward pass
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        epoch_loss += loss.item()

    average_loss = epoch_loss / len(train_loader)

    print(
        f"Epoch [{epoch + 1}/{EPOCHS}] "
        f"- Loss: {average_loss:.6f}"
    )

Epoch [1/5] - Loss: 0.105903
Epoch [2/5] - Loss: 0.061921
Epoch [3/5] - Loss: 0.026925
Epoch [4/5] - Loss: 0.008434
Epoch [5/5] - Loss: 0.004495


In [6]:
from pathlib import Path

MODEL_PATH = (
    PROJECT_ROOT
    / "models"
    / CATEGORY
    / "autoencoder.pth"
)

MODEL_PATH.parent.mkdir(parents=True, exist_ok=True)

torch.save(
    model.state_dict(),
    MODEL_PATH
)

print("Model saved to:")
print(MODEL_PATH)

Model saved to:
X:\VScode\Artificial_intelligence_n_Machine_learning\VisionInspect\models\bottle\autoencoder.pth
